# Modifying ABCMB

In this script we demonstrate how to efficiently implement a new physics fluid species to the cosmological model.

TODO: Discuss how to add new species, how to handle delta_idx? Not by the user?

## Setup

In [3]:
import sys
sys.path.append('..')

from ABCMB.main import Model
import ABCMB.spectrum as spectrum
from ABCMB.AbstractSpecies import AbstractFluid, AbstractPerturbedFluid

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

## Example: SIDR

We will first look at the strongly self-interacting dark radiation model. This scenario can be realised by a secluded dark sector of massless particles, whose self-interaction remains effective throughout cosmological history $\left(\frac{\Gamma}{H}\gg 1\right)$. Such interaction reduces the full Boltzmann hierarchy required for a free-streaming species down to the $\ell=0, 1$ modes.

Here, the user can define a new SIDR class, which inherits from ABCMB's template class, AbstractPerturbedFluid.
For the time being we specify the strict=True flag. We will discuss this option and its implications later on in this notebook.

First we will look at the fully implemented class, then break down the necessary components.

In [4]:
class SIDR(AbstractPerturbedFluid, strict=True):

    # Index corresponding to position of species in the full differential equations
    delta_idx : int

    # Number of modes in the Boltzmann Hierarchy
    num_ell_modes = 2 # Non free-streaming species only require density and velocity perturbations
    
    def rho(self, lna, params):
        """
        Energy density at log scale factor lna.
        Should be in units of eV/cm^3.

        Here it is parameterized in terms of N_IDR, the effective number
        of extra neutrino species. 
        """
        # Energy density in one neutrino species
        rho1nu = params["omega_nu"]/params["N_ur"] * (3.*cnst.H0_over_h**2/8./jnp.pi/cnst.G) / jnp.exp(lna)**4
        return params["N_IDR"] * rho1nu

    def P(self, lna, params):
        """
        Pressure at log scale factor lna.
        For fully relativistic species this is simply rho/3.
        """
        return self.rho(lna, params)/3.

    def cs2(self, lna, params):
        """
        Sound speed squared.
        For fully relativistic species cs2 = 1/3.
        """
        return 1./3.

    def y_ini(self, k, tau_ini, om, args):
        """
        Adiabatic superhorizon initial conditions for SIDR.
        For all relativistic species they are matched to neutrinos.
        """
        params = args
        R_nu = params['R_nu']

        delta = - (k*tau_ini)**2/3. * (1.-om*tau_ini/5.)
        theta = - k*(k*tau_ini)**3/36./(4.*R_nu+15.) \
                * (4.*R_nu+11.+12.-3.*(8.*R_nu**2+50.*R_nu+275.)/20./(2.*R_nu+15.)*tau_ini*om)
        return jnp.array([delta, theta])

    def y_prime(self, k, lna, metric_h_prime, metric_eta_prime, y, args):
        """
        Derivatives of the SIDR perturbations w.r.t. lna.
        """
        BG, params = args
        aH = BG.aH(lna)

        # First find the delta and theta that belong to SIDR
        delta = y[self.delta_idx]
        theta = y[self.delta_idx+1]
        
        delta_prime = -4./3./aH*theta - 2./3.*metric_h_prime
        theta_prime = k**2/aH*delta/4.

        return jnp.array([delta_prime, theta_prime])

### Understanding strict=True

TODO: explain equinox.Module inheritance philosophy.

## Example: $w_0w_a$ (Dynamical Dark Energy)

The user might also wish to implement a species without any inhomogeneity. Such a species would only contribute to the background portion of the Friedmann equation, i.e. modify Hubble in an interesting way. 

One such model, the dynamical dark energy, has gained much attention in light of the recent DESI data. We demonstrate here how to implement such a model, and similar models that do not require a perturbation Boltzmann hierarchy. 

Since the model has no perturbations, it is no longer necessary to implement the functions y_ini() and y_prime(). This means the class w0wa should not inherit from AbstractPerturbedFluid, which requires that all children implement these functions. Instead, we will go one step up the chain and inherit from the perturbations-free parent class, AbstractFluid.

In [6]:
class w0wa(AbstractFluid, strict=True):

    # Note: w0wa class no longer has fields delta_idx and num_ell_modes.
    
    def rho(self, lna, params):
        """
        Energy density at log scale factor lna.
        Should be in units of eV/cm^3.

        Parameterized as the redshift behavior based on its energy density today.
        rho = rho0 x a^{-3(1+w)}
        """
        # Energy density today
        rho0 = params["omega_Lambda"] * (3.*cnst.H0_over_h**2/8./jnp.pi/cnst.G)
        a = jnp.exp(lna)
        w = params["w0DE"] + (1-a)*params["waDE"]
        
        return rho0 * a**(-3.*(1.+w))

    def P(self, lna, params):
        """
        Pressure at log scale factor lna.
        """
        a = jnp.exp(lna)
        w = params["w0DE"] + (1-a)*params["waDE"]
        return w*self.rho(lna, params)

    def cs2(self, lna, params):
        """
        Sound speed squared.
        This is not used, but for good programming structure it must be instantiated.
        Set to some trivial value. 
        """
        return 0.

## Example: Interacting DM-DR

## Template: Define your own species!